# 19 · L4 Model Development Capstone

这个 capstone 把前面的接口串成一条最小的 L4 模型开发闭环：

\[
\text{scenario} \rightarrow \text{sensor/model signals} \rightarrow \text{prediction/planning} \rightarrow \text{safety gate} \rightarrow \text{metrics}.
\]

目标不是声称一个合成实验“实现了 L4”，而是练习岗位真正需要的工程证据：

- 一组有 ODD 标签的场景；
- 一个可替换的 learned-policy 接口；
- 一个独立 safety gate；
- open-loop 与 closed-loop 风险指标；
- latency、fallback、舒适性和失败案例报告。

你最后应把本 notebook 的报告替换成 nuPlan/NAVSIM/CARLA 或公司内部 scenario runner 的输出。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

@dataclass(frozen=True)
class Episode:
    episode_id: str
    weather: str
    density: str
    sensor_fault: str
    initial_gap_m: float
    obstacle_speed_mps: float
    sensor_health: float
    localization_error_m: float
    odd_ok: bool
    base_latency_ms: float


episodes = [
    Episode("urban_nominal_01", "clear", "medium", "none", 28, 4.0, 0.96, 0.25, True, 48),
    Episode("urban_lidar_dropout", "clear", "high", "lidar_dropout", 20, 3.0, 0.58, 0.65, True, 55),
    Episode("rain_relocalization", "rain", "medium", "gnss_outage", 24, 4.0, 0.78, 1.10, True, 63),
    Episode("construction_odd_exit", "clear", "high", "map_stale", 18, 2.0, 0.62, 1.55, False, 71),
    Episode("rare_close_cut_in", "night", "high", "camera_glare", 10, 1.0, 0.52, 0.90, True, 67),
]


## Part A — 可替换的 policy 与 safety gate

`naive` 代表只根据 nominal training distribution 生成速度，`guarded` 则使用健康度、定位误差、ODD 和 gap 触发独立保护。真实项目中，policy 可以是 Transformer、diffusion/flow matching 或 VLA，gate 仍应有独立验证路径。


In [ ]:
def run_episode(ep, policy="guarded", dt=0.1, horizon_s=8.0, seed=4):
    rng = np.random.default_rng(seed)
    steps = int(horizon_s / dt)
    ego_x, ego_speed = 0.0, 7.0
    obstacle_x = ep.initial_gap_m
    rows = []
    for step in range(steps):
        t = step * dt
        obstacle_speed = ep.obstacle_speed_mps if t < 2.8 else max(0.0, ep.obstacle_speed_mps - 3.0)
        gap = obstacle_x - ego_x
        risk = (gap < 12.0 or ep.sensor_health < 0.70 or ep.localization_error_m > 0.80 or not ep.odd_ok)
        if policy == "naive":
            target_speed = 7.0
            action = "learned_policy"
        elif risk:
            target_speed = max(0.0, min(3.0, obstacle_speed - 0.5))
            action = "minimal_risk" if gap < 7.0 or not ep.odd_ok else "degraded"
        else:
            target_speed = 7.0
            action = "learned_policy"
        acceleration = np.clip((target_speed - ego_speed) * 1.6, -4.0, 2.0)
        ego_speed = max(0.0, ego_speed + acceleration * dt)
        ego_x += ego_speed * dt
        obstacle_x += obstacle_speed * dt
        rows.append({"time_s": t, "gap": obstacle_x - ego_x, "ego_speed": ego_speed,
                     "acceleration": acceleration, "action": action,
                     "latency_ms": ep.base_latency_ms + rng.lognormal(-2.2, 0.25)})
    frame = pd.DataFrame(rows)
    acceleration = frame["acceleration"].to_numpy()
    jerk = np.diff(acceleration, prepend=acceleration[0]) / dt
    fallback = frame["action"].ne("learned_policy")
    return frame, {
        "episode_id": ep.episode_id,
        "policy": policy,
        "collision": bool((frame["gap"] < 2.0).any()),
        "min_gap_m": float(frame["gap"].min()),
        "progress_m": float((frame["ego_speed"] * dt).sum()),
        "fallback_seconds": float(fallback.sum() * dt),
        "max_jerk_mps3": float(np.abs(jerk).max()),
        "p95_latency_ms": float(frame["latency_ms"].quantile(0.95)),
        "odd_violation": not ep.odd_ok,
    }


reports = []
traces = {}
for policy in ["naive", "guarded"]:
    for ep in episodes:
        traces[(policy, ep.episode_id)], report = run_episode(ep, policy=policy)
        reports.append(report)
report_df = pd.DataFrame(reports)
display(report_df)


In [ ]:
summary = report_df.groupby("policy").agg(
    collision_rate=("collision", "mean"),
    mean_min_gap_m=("min_gap_m", "mean"),
    mean_fallback_s=("fallback_seconds", "mean"),
    p95_latency_ms=("p95_latency_ms", "max"),
    mean_max_jerk=("max_jerk_mps3", "mean"),
)
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for policy in ["naive", "guarded"]:
    for ep in episodes:
        frame = traces[(policy, ep.episode_id)]
        axes[0].plot(frame["time_s"], frame["gap"], alpha=0.45, label=policy if ep is episodes[0] else None)
axes[0].axhline(2.0, color="red", linestyle="--", label="collision threshold")
axes[0].set(xlabel="time / s", ylabel="gap / m", title="Per-episode gap traces")
axes[0].legend()
summary["collision_rate"].plot(kind="bar", ax=axes[1], color=["#b91c1c", "#15803d"])
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("collision rate")
axes[1].set_title("Guarded policy must be evaluated on safety and cost")
plt.show()


## Part B — 作品集交付物

这个 capstone 不应以“guarded policy 的分数更高”结束。请输出：

1. ODD 说明和场景矩阵；
2. policy/safety gate 的输入输出契约；
3. naive 与 guarded 的 collision、fallback、progress、jerk、p95 latency 对比；
4. 至少一个 failure replay，指出是感知、定位、规划、系统健康还是评测设计导致；
5. 一个下一步真实数据接入计划，包括数据许可、坐标/时间同步、scenario ID 和 regression gate。

### 练习

- 增加一个 `open_loop` 指标，并解释它为什么不能替代 closed-loop collision rate；
- 把 scenario coverage 按 weather、density、sensor_fault 分组；
- 对 latency budget 增加 p99 和 watchdog violation；
- 把 safety gate 拆成可以独立单元测试的函数，并构造至少三个 adversarial case。


In [ ]:
coverage = pd.DataFrame([
    {"weather": ep.weather, "density": ep.density, "sensor_fault": ep.sensor_fault}
    for ep in episodes
]).value_counts().rename("episodes").reset_index()
display(coverage)
print("portfolio checklist:")
print("- scenario matrix:", len(episodes), "episodes")
print("- policies compared: ", report_df["policy"].unique().tolist())
print("- safety metric: collision rate + minimum gap + fallback duration")
print("- runtime metric: p95 latency; add p99 and watchdog in your submission")


## 边界声明

该 notebook 是一个接口和验证骨架，不是自动驾驶系统认证证据。真正的 L4 项目还需要真实传感器、车辆动力学、地图、交通参与者、硬件 runtime、系统冗余和组织级 safety case。
